In [5]:

import sys
sys.path.append("../src")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from evaluation import time_split, evaluate, diff_ci

df = pd.read_csv("../data/LUCK_target.csv", index_col="Date", parse_dates=True)
print(df.shape)
print(df.columns.tolist())



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(2884, 9)
['Close', 'High', 'Low', 'Open', 'Volume', 'next_close', 'next_return', 'target', 'ret']


In [2]:
dev, test = time_split(df)

print("dev :", dev.index.min().date(), "to", dev.index.max().date(), len(dev))
print("test:", test.index.min().date(), "to", test.index.max().date(), len(test))
print("dev up share:", round(dev["target"].mean(), 4))
print("dev lag-1 correlation:", round(dev["ret"].corr(dev["ret"].shift(1)), 4))
print(dev.groupby(dev.index.year)["target"].mean().round(3))


dev : 2015-01-01 to 2023-10-13 2162
test: 2023-10-17 to 2026-10-12 721
dev up share: 0.4741
dev lag-1 correlation: 0.1315
Date
2015    0.472
2016    0.516
2017    0.452
2018    0.463
2019    0.465
2020    0.496
2021    0.447
2022    0.444
2023    0.524
Name: target, dtype: float64


In [3]:
majority = int(dev["target"].mean() > 0.5)
print("majority class from dev:", majority)

y_test = test["target"].to_numpy()
pred_A = np.full(len(test), majority)
pred_B = (test["ret"] > 0).astype(int).to_numpy()

print(test[["Close", "ret"]].head(3))
print(pred_B[:3])


majority class from dev: 0
                 Close       ret
Date                            
2023-10-17  115.662708  0.008048
2023-10-18  116.344055  0.005891
2023-10-19  116.406531  0.000537
[1 1 1]


In [4]:
prob_A = np.full(len(test), dev["target"].mean())

results = pd.DataFrame([
    evaluate(y_test, pred_A, "A: majority class", prob_A),
    evaluate(y_test, pred_B, "B: previous direction"),
])
print(results)
print("Confusion matrix A (rows = truth, cols = prediction):")
print(confusion_matrix(y_test, pred_A))
print("Confusion matrix B:")
print(confusion_matrix(y_test, pred_B))
print("accuracy(B) - accuracy(A): diff, ci_low, ci_high =", diff_ci(y_test, pred_B, pred_A))


                  method  n_test  accuracy          95% CI  balanced_acc  \
0      A: majority class     721    0.4938  0.457 to 0.530        0.5000   
1  B: previous direction     721    0.5354  0.499 to 0.572        0.5353   

   precision  recall  log_loss  
0      0.000  0.0000    0.6951  
1      0.541  0.5425       NaN  
Confusion matrix A (rows = truth, cols = prediction):
[[356   0]
 [365   0]]
Confusion matrix B:
[[188 168]
 [167 198]]
accuracy(B) - accuracy(A): diff, ci_low, ci_high = (np.float64(0.0416), np.float64(-0.0125), np.float64(0.0929))


Baseline B achieved higher test accuracy (53.54%) than Baseline A (49.38%), an improvement of 4.16 percentage points. However, the paired bootstrap 95% confidence interval for the accuracy difference included zero (-1.25 to 9.29 percentage points). Therefore, the improvement is not statistically conclusive.